# LNP-Transfection: Colab Runner

This notebook runs the full pipeline headlessly on Google Colab, predicting
**cell-specific transfection efficiency** from LNP SMILES strings using the
AGILE dataset format (`SMILES`, `Transfection`):
1. Clone the GitHub repository (edit the URL below).
2. Install dependencies.
3. Run feature generation (`data_processing.py`).
4. Train the XGBoost transfection model (`train.py`).
5. Generate SHAP interpretability plots (`evaluate.py`).

**Before running:** place your raw dataset at `data/raw/dataset.csv` as a CSV
with `SMILES` and `Transfection` columns.

In [ ]:
# @title 1. Configuration
REPO_URL = "https://github.com/Hich00b/lnp-transfection-ml.git"  # @param {type:"string"}
REPO_DIR = "lnp-transfection-ml"  # @param {type:"string"}
RAW_CSV = "data/raw/dataset.csv"  # @param {type:"string"}
print(f"Repository: {REPO_URL}
Clone dir:  {REPO_DIR}
Raw CSV:    {RAW_CSV}")

In [ ]:
# @title 2. Clone the repository
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}

In [ ]:
# @title 3. Install dependencies
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# @title 4. Generate features (Morgan fingerprints + descriptors)
!python src/data_processing.py --input {RAW_CSV} --out-dir data/processed

In [ ]:
# @title 5. Train the XGBoost transfection model (scaffold split)
# Transfection is the default target -- no --target flag needed.
# For K-fold cross-validation instead, add: --cv 5
!python src/train.py --features data/processed/features.csv --targets data/processed/targets.csv

In [ ]:
# @title 6. SHAP interpretability analysis
!python src/evaluate.py --model models/xgb_transfection.pkl --features data/processed/features.csv

In [ ]:
# @title 7. Display the SHAP summary plot
from IPython.display import Image, display
display(Image("models/shap_summary_transfection.png"))
display(Image("models/shap_descriptors_transfection.png"))

## Outputs

| Artefact | Location |
|---|---|
| Feature matrix | `data/processed/features.csv` |
| Targets | `data/processed/targets.csv` |
| Trained model | `models/xgb_transfection.pkl` |
| SHAP beeswarm | `models/shap_summary_transfection.png` |
| Descriptor importance | `models/shap_descriptors_transfection.png` |

Defaults target the `Transfection` column (AGILE format). To predict another
numeric target, pass `--target <column>` to `train.py` and `evaluate.py`.